In [0]:
# ==============================================================================
# CONTROL VARIABLES
# ==============================================================================
TARGET_ENV = "prod"  # "dev" or "prod"

In [0]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================
import time
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Catalog derivation
CATALOG = "8_dev" if TARGET_ENV == "dev" else "4_prod"
LOG_CATALOG = "6_mgmt"

# --- Org ID lists ---
BARTS_ORGS = [
    873843, 8367658, 669849, 9073614, 2681833, 4401825, 3203824, 2681830,
    8061679, 669848, 8467812, 2681824, 2619824, 2681827, 3203825, 691988,
    3125827, 8061682, 8061694, 2641824, 2641827, 669847, 8056759, 8061685,
    2641830, 3201824, 691989, 669845, 669843, 8061691, 669846, 3199824,
    669850, 6333825, 669844, 8397458, 8152502, 671843, 613843
]

BHRUT_ORGS = [
    9161976, 9163579, 9161983, 723896, 9161987, 9161988, 9163583, 9161989
]

ALL_ORGS = BARTS_ORGS + BHRUT_ORGS

# --- Schemas ---
SCHEMA_RAW = f"{CATALOG}.raw"
SCHEMA_TMP = f"{CATALOG}.tmp"
SCHEMA_STG = f"{CATALOG}.staging"

# --- Table FQNs (must match classification notebook) ---
TRUST_MAP_TBL   = f"{SCHEMA_TMP}.org_to_trust_map"
ENC_ORG_TBL     = f"{SCHEMA_TMP}.sw_enc_org_map"
HUB_TBL         = f"{SCHEMA_TMP}.sw_mapping_hub"
CHANGED_ENC_TBL = f"{SCHEMA_TMP}.sw_changed_encounters"
CONTROL_TBL     = f"{SCHEMA_TMP}.incr_updt_trust_control"

LOG_TABLE = f"{LOG_CATALOG}.logs.trust_maintenance_log"

# --- Skip lists (tables that should not have BHRUT rows deleted) ---
SKIP_BHRUT_DELETE = {"mill_person", "mill_address", "mill_person_alias", "mill_person_name",
                     "mill_person_patient", "mill_person_info", "mill_person_org_reltn",
                     "mill_person_person_reltn", "mill_person_prsnl_reltn", "mill_prsnl",
                     "mill_prsnl_alias", "mill_prsnl_org_reltn", "mill_prsnl_reltn"}
SKIP_NULL_CLEAN = {"mill_person", "mill_address"}

print("=" * 70)
print(f"  TRUST MAINTENANCE INCREMENTAL")
print(f"  TARGET_ENV : {TARGET_ENV}")
print(f"  CATALOG    : {CATALOG}")
print(f"  Timestamp  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

In [0]:
# ==============================================================================
# HELPERS
# ==============================================================================

def time_op(label, func, *args, **kwargs):
      t0 = time.time()
      err_msg = None
      rows_affected = None
      try:
          print(f"\n{'─' * 60}")
          print(f"▶ START: {label}")
          result = func(*args, **kwargs)
          if isinstance(result, int) and not isinstance(result, bool):
              rows_affected = result
          elapsed = time.time() - t0
          print(f"✔ DONE : {label}  [{elapsed:.1f}s]")
          return result
      except Exception as e:
          elapsed = time.time() - t0
          err_msg = f"{type(e).__name__}: {str(e).splitlines()[0][:500]}".replace("'", "''")
          print(f"✘ FAIL : {label}  [{elapsed:.1f}s] — {err_msg}")
          raise
      finally:
          try:
              rows_sql  = str(rows_affected) if rows_affected is not None else "NULL"
              err_sql   = f"'{err_msg}'" if err_msg is not None else "NULL"
              spark.sql(f"""
                  INSERT INTO {LOG_TABLE} VALUES (
                      current_timestamp(),
                      '{label}',
                      NULL,
                      {rows_sql},
                      {elapsed:.3f},
                      {err_sql}
                  )
              """)
          except Exception:
              pass


def get_table_columns(table_fqn):
    """Return list of column names (uppercase) for a Delta table."""
    return [c.name.upper() for c in spark.table(table_fqn).schema.fields]


def discover_raw_tables_with_trust():
    """Find all raw tables that have a Trust column. Limited to mill_* tables."""
    all_raw = [r.tableName for r in spark.sql(f"SHOW TABLES IN {SCHEMA_RAW}").collect()]
    trust_tables = []
    for tbl in all_raw:
        if not tbl.startswith("mill_"):
            continue
        try:
            cols = get_table_columns(f"{SCHEMA_RAW}.{tbl}")
            if "TRUST" in cols:
                trust_tables.append(tbl)
        except Exception:
            pass
    return trust_tables

In [0]:
# ==============================================================================
# CELL 4: REFRESH TRUST MAP
# ==============================================================================

def refresh_trust_map():
    """Rebuild the trust-org mapping table from the canonical org lists."""
    rows = []
    for org_id in BARTS_ORGS:
        rows.append((org_id, "Barts"))
    for org_id in BHRUT_ORGS:
        rows.append((org_id, "BHRUT"))

    df = spark.createDataFrame(rows, ["organization_id", "trust"])
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TRUST_MAP_TBL)
    print(f"  Trust map refreshed: {df.count()} orgs written to {TRUST_MAP_TBL}")

In [0]:
# ==============================================================================
# CELL 5: PROPAGATE ORG REMAPPING
# ==============================================================================

def propagate_org_remapping():
    """
    Check sw_changed_encounters for encounters whose org mapping changed recently.
    Look up the new trust via enc_org -> trust_map, then MERGE into raw tables.
    """
    cutoff = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")

    try:
        changed_df = spark.sql(f"""
            SELECT DISTINCT ce.ENCNTR_ID, tm.trust
            FROM {CHANGED_ENC_TBL} ce
            JOIN {ENC_ORG_TBL} eo ON ce.ENCNTR_ID = eo.ENCNTR_ID
            JOIN {TRUST_MAP_TBL} tm ON eo.ORGANIZATION_ID = tm.organization_id
            WHERE ce.detected_at >= '{cutoff}'
        """)
        change_count = changed_df.count()
        print(f"  Encounters with trust change since {cutoff}: {change_count:,}")
    except Exception as e:
        print(f"  Could not read changed encounters: {e}")
        print("  Skipping MERGE propagation.")
        return

    if change_count == 0:
        print("  No trust changes to propagate.")
        return

    changed_df.createOrReplaceTempView("_changed_trusts")

    raw_tables = discover_raw_tables_with_trust()
    for tbl_name in raw_tables:
        tbl_fqn = f"{SCHEMA_RAW}.{tbl_name}"
        cols = get_table_columns(tbl_fqn)
        if "ENCNTR_ID" not in cols:
            continue

        try:
            merge_sql = f"""
                MERGE INTO {tbl_fqn} AS target
                USING _changed_trusts AS src
                ON target.ENCNTR_ID = src.ENCNTR_ID
                WHEN MATCHED AND (target.Trust != src.trust OR target.Trust IS NULL) THEN
                    UPDATE SET target.Trust = src.trust
            """
            result = spark.sql(merge_sql)
            metrics = result.first()
            updated = metrics["num_affected_rows"] if "num_affected_rows" in result.columns else "?"
            print(f"  {tbl_name}: {updated} rows updated")
        except Exception as e:
            print(f"  {tbl_name}: FAILED — {str(e).split(chr(10))[0][:100]}")

In [0]:
# ==============================================================================
# CELL 6: CLEANUP BHRUT ROWS
# ==============================================================================

def cleanup_bhrut_rows():
    """DELETE FROM raw tables WHERE Trust = 'BHRUT' for eligible tables."""
    raw_tables = discover_raw_tables_with_trust()

    for tbl_name in raw_tables:
        if tbl_name in SKIP_BHRUT_DELETE:
            print(f"  {tbl_name}: skipped (protected)")
            continue

        tbl_fqn = f"{SCHEMA_RAW}.{tbl_name}"
        before_count = spark.table(tbl_fqn).filter(F.upper(F.col("Trust")) == "BHRUT").count()
        if before_count == 0:
            continue

        spark.sql(f"DELETE FROM {tbl_fqn} WHERE UPPER(Trust) = 'BHRUT'")
        print(f"  {tbl_name}: deleted {before_count:,} BHRUT rows")

In [0]:
# ==============================================================================
# CELL 7: DEEP CLEAN NULL TRUST
# ==============================================================================

def deep_clean_null_trust():
    """
    For each raw table with NULL trust values:
      1. Try to resolve via org mapping (ORGANIZATION_ID -> trust).
      2. Fall back to encounter-level resolution (ENCNTR_ID -> enc_org -> trust).
    """
    raw_tables = discover_raw_tables_with_trust()

    for tbl_name in raw_tables:
        if tbl_name in SKIP_NULL_CLEAN:
            print(f"  {tbl_name}: skipped (in skip list)")
            continue

        tbl_fqn = f"{SCHEMA_RAW}.{tbl_name}"
        cols = get_table_columns(tbl_fqn)

        null_count = spark.table(tbl_fqn).filter(F.col("Trust").isNull()).count()
        if null_count == 0:
            continue

        print(f"  {tbl_name}: {null_count:,} NULL trust rows...")
        resolved = 0

        # Strategy 1: Direct org resolution
        if "ORGANIZATION_ID" in cols:
            org_sql = f"""
                MERGE INTO {tbl_fqn} AS target
                USING {TRUST_MAP_TBL} AS tm
                ON target.ORGANIZATION_ID = tm.organization_id
                WHEN MATCHED AND target.Trust IS NULL THEN
                    UPDATE SET target.Trust = tm.trust
                    {"  , target.ADC_UPDT = current_timestamp()" if "ADC_UPDT" in cols else ""}
            """
            spark.sql(org_sql)
            remaining = spark.table(tbl_fqn).filter(F.col("Trust").isNull()).count()
            resolved_by_org = null_count - remaining
            resolved += resolved_by_org
            if resolved_by_org > 0:
                print(f"    Org resolution: {resolved_by_org:,} rows fixed")
            null_count = remaining

        # Strategy 2: Encounter-based resolution
        if null_count > 0 and "ENCNTR_ID" in cols:
            enc_sql = f"""
                MERGE INTO {tbl_fqn} AS target
                USING (
                    SELECT DISTINCT eo.ENCNTR_ID, tm.trust
                    FROM {ENC_ORG_TBL} eo
                    JOIN {TRUST_MAP_TBL} tm ON eo.ORGANIZATION_ID = tm.organization_id
                ) AS enc_trust
                ON target.ENCNTR_ID = enc_trust.ENCNTR_ID
                WHEN MATCHED AND target.Trust IS NULL THEN
                    UPDATE SET target.Trust = enc_trust.trust
                    {"  , target.ADC_UPDT = current_timestamp()" if "ADC_UPDT" in cols else ""}
            """
            spark.sql(enc_sql)
            remaining = spark.table(tbl_fqn).filter(F.col("Trust").isNull()).count()
            resolved_by_enc = null_count - remaining
            resolved += resolved_by_enc
            if resolved_by_enc > 0:
                print(f"    Encounter resolution: {resolved_by_enc:,} rows fixed")
            null_count = remaining

        print(f"    Total resolved: {resolved:,} | Still NULL: {null_count:,}")

In [0]:
# ==============================================================================
# CELL 8: OPTIMIZE TABLES
# ==============================================================================

def optimize_tables():
    """Run OPTIMIZE and VACUUM on hub/lookup tables."""
    tables_to_optimize = [TRUST_MAP_TBL, ENC_ORG_TBL, HUB_TBL]
    for tbl in tables_to_optimize:
        try:
            spark.sql(f"OPTIMIZE {tbl}")
            spark.sql(f"VACUUM {tbl}")
            print(f"  {tbl}: optimized + vacuumed")
        except Exception as e:
            print(f"  {tbl}: {str(e).split(chr(10))[0][:100]}")

In [0]:
# ==============================================================================
# CELL 9: STAGING HEALTH REPORT
# ==============================================================================

def staging_health_report():
    """Report row counts, oldest staged_at, and classification attempts per staging table."""
    print(f"\n{'=' * 70}")
    print(f"  STAGING HEALTH REPORT")
    print(f"{'=' * 70}")

    try:
        staging_tables = sorted([
            r.tableName for r in spark.sql(f"SHOW TABLES IN {SCHEMA_STG}").collect()
            if r.tableName.startswith("mill_")
        ])
    except Exception as e:
        print(f"  Could not list staging tables: {e}")
        return

    if not staging_tables:
        print("  No staging tables found.")
        return

    warnings = []
    total_rows = 0

    for tbl_name in staging_tables:
        tbl_fqn = f"{SCHEMA_STG}.{tbl_name}"
        try:
            stats = spark.table(tbl_fqn).agg(
                F.count("*").alias("rows"),
                F.min("_staged_at").alias("oldest"),
                F.max("_classification_attempts").alias("max_att"),
                F.round(F.avg("_classification_attempts"), 1).alias("avg_att"),
            ).first()

            row_count = stats["rows"]
            total_rows += row_count

            if row_count == 0:
                continue

            print(
                f"  {tbl_name:<40} rows={row_count:<10,} "
                f"oldest={str(stats['oldest'])[:19]:<22} "
                f"max_att={stats['max_att']}  avg_att={stats['avg_att']}"
            )

            if row_count > 1000 and (stats["max_att"] or 0) >= 3:
                warnings.append(f"  WARNING: {tbl_name} has {row_count:,} rows with max {stats['max_att']} attempts — may be stuck")

        except Exception as e:
            print(f"  {tbl_name:<40} ERROR: {str(e)[:80]}")

    print(f"\n  Total rows across all staging tables: {total_rows:,}")

    if warnings:
        print(f"\n{'!' * 70}")
        for w in warnings:
            print(w)
        print(f"{'!' * 70}")
    else:
        print("  No stuck records detected.")

In [0]:
# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

overall_start = time.time()

time_op("1. Refresh trust map",           refresh_trust_map)
time_op("2. Propagate org remapping",     propagate_org_remapping)
time_op("3. Cleanup BHRUT rows",          cleanup_bhrut_rows)
time_op("4. Deep clean NULL trust",       deep_clean_null_trust)
time_op("5. Optimize lookup tables",      optimize_tables)
time_op("6. Staging health report",       staging_health_report)

overall_elapsed = time.time() - overall_start

print(f"\n{'=' * 70}")
print(f"  ALL STEPS COMPLETE")
print(f"  TARGET_ENV : {TARGET_ENV}")
print(f"  CATALOG    : {CATALOG}")
print(f"  Total time : {overall_elapsed:.1f}s ({overall_elapsed/60:.1f}m)")
print(f"{'=' * 70}")